### Text Embedding for Retrieval:  
We have a sentence or phrase and get its embedding from a pretrained model.

In [50]:
#!pip install -U sentence-transformers

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import random

In below cell, we initialize a specialized bi-encoder model designed by Microsoft.
Internally, this single line of code triggers a complex pipeline that transforms raw text into a fixed-length numerical vector (an embedding).

In [16]:
model = SentenceTransformer('intfloat/e5-small-v2')

In [38]:
query_text="query: feather coat"
doc_text="passage: Fishing bait accessory, feathered jig lure. Unisex, for cod, rockfish, baitfish, and more. Feathers, highest quality, sizes 3/0 and 5/0. P-Line, fishing context."

The normalize_embeddings Parameter:  
If False (Default): You get the direct output of the mean-pooling layer. The vector lengths (magnitudes) will vary. To compare these, you must use Cosine Similarity.  
If True: The library applies $L2$ normalization. Every vector is scaled to have a length of exactly $1$. In this case, Dot Product and Cosine Similarity become mathematically identical.

In [44]:
query_embed=model.encode(query_text,normalize_embeddings=True)
doc_embed=model.encode(doc_text,normalize_embeddings=True)

Since SentenceTransformer abstracts the tokenization process, you can't see the tokens by just calling .encode(). To see exactly how your text is being chopped up, you need to access the underlying tokenizer object. The model maintains a map of tokens to ID. The transformation from a word like "garden" to an ID like 3842 happens through a process called Vocabulary Mapping. 1. This is a Pre-compiled Vocabulary of  ~30,000 most common words and sub-word pieces. Every single one of these was assigned a fixed index or ID.

0: [PAD]
101: [CLS] (The "Start" marker)
102: [SEP] (The "Separator" marker)
3842: garden
4638: shoes
1. Special Tokens: You'll see [CLS] (start of sequence) and [SEP] (end of sequence).
2. Subword Splitting: Words the model doesn't fully recognize might be split. For exampl e, "tokenization" might become token, ##iz, ##ation.

2. The WordPiece Algorithm
The model doesn't just look up whole words. If it sees a word it doesn't know, it breaks it into pieces it does know. This is why you sometimes see ## in your tokens.

Example: "Tokenizing"

Is "tokenizing" in the vocab? No.
Is "token" in the vocab? Yes. (Assign ID for token)
Is "izing" in the vocab? No.
Is "##iz" in the vocab? Yes. (Assign ID)
Is "##ing" in the vocab? Yes. (Assign ID)

Since SentenceTransformer encapsulates
There are two ways to do this: using the SentenceTransformer object you already have, or using the AutoTokenizer from the transformers library.

1. Using the existing model object
The SentenceTransformer class stores the tokenizer inside its first module (the Transformer layer).

**Why IDs matter to the Model**  
The model (the Neural Network) cannot "read" text. It only understands numbers. However, it doesn't even use the ID 3842 for math. Instead, the ID acts as an address.

The model goes to "address" 3842 in its Embedding Matrix and pulls out a vector of 384 floating-point numbers. That vector is what actually enters the Transformer layers.

In [45]:
# Access the tokenizer from the model's first module
tokenizer = model.tokenizer

# 1. See the Token IDs
tokens = tokenizer.encode(query_text)
print(f"Token IDs: {tokens}")

# 2. See the actual String Tokens (the "WordPieces")
tokens_visual = tokenizer.tokenize(query_text)
print(f"Tokens:    {tokens_visual}")

# 3. See the mapping of ID -> Token
for token_id in tokens:
    print(f"{token_id:<8} -> {tokenizer.decode([token_id])}")

Token IDs: [101, 23032, 1024, 15550, 5435, 102]
Tokens:    ['query', ':', 'feather', 'coat']
101      -> [CLS]
23032    -> query
1024     -> :
15550    -> feather
5435     -> coat
102      -> [SEP]


In [46]:
inputs = tokenizer(query_text, padding=True, truncation=True, return_tensors="pt")
print(inputs)
# Output: {'input_ids': tensor([...]), 'attention_mask': tensor([...])}

{'input_ids': tensor([[  101, 23032,  1024, 15550,  5435,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


In [47]:
score = model.similarity(query_embed, doc_embed)

In [48]:
print(score)

tensor([[0.8395]])


### TO BE CONTINUED